In [1]:
import sys
from pathlib import Path

import pandas as pd
import sqlite3

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src").exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATABASE_FILE, DATA_CONTROL_CENTER_FILE


# Phase 4 - Validation

Le brief demande six contrôles et trois livrables : rapport, KPI, dashboard.

Cette validation lit le dernier passage enregistré dans SQLite, puis vérifie que le classeur en est l'export. Elle lit aussi les fichiers traités. Elle échoue si un contrôle du brief est absent, si un jeu activé n'est pas noté, si le classeur ne reprend pas le score enregistré, ou si un fichier traité contredit les règles de préparation.


In [2]:
REQUIRED_CONTROLS = [
    "Valeurs manquantes",
    "Doublons",
    "Valeurs aberrantes",
    "Formats incorrects",
    "Cohérence des devises",
    "Chronologie",
]

connection = sqlite3.connect(DATABASE_FILE)
run = pd.read_sql(
    """
    SELECT *
    FROM quality_run
    ORDER BY execution_date DESC
    LIMIT 1
    """,
    connection,
)
assert len(run) == 1, "Aucun passage qualité n'est enregistré."
run_id = run.loc[0, "run_id"]

controls = pd.read_sql(
    """
    SELECT dataset_id, control_id, control_name, status, issues, records_checked
    FROM quality_control_result
    WHERE run_id = ?
    ORDER BY dataset_id, control_id
    """,
    connection,
    params=(run_id,),
)
metrics = pd.read_sql(
    """
    SELECT dataset_id, metric_name, metric_value, metric_status, applicable
    FROM quality_metric_result
    WHERE run_id = ?
    """,
    connection,
    params=(run_id,),
)
registry = pd.read_sql(
    """
    SELECT dataset_id
    FROM dataset_registry
    WHERE active = 1 AND quality_enabled = 1
    """,
    connection,
)
connection.close()

scored = controls.loc[controls["dataset_id"].isin(registry["dataset_id"])]
expected = set(registry["dataset_id"])
assert set(scored["dataset_id"]) == expected, (set(scored["dataset_id"]), expected)
assert set(metrics["dataset_id"]) == expected
assert set(scored["control_name"]) == set(REQUIRED_CONTROLS)
assert scored.groupby("dataset_id")["control_name"].nunique().eq(6).all()
assert not scored["status"].eq("FAIL").any()

currency = controls[
    (controls["dataset_id"] == "DS-002")
    & (controls["control_name"] == "Cohérence des devises")
]
assert len(currency) == 1
assert currency["status"].iloc[0] != "FAIL"

assert run.loc[0, "global_quality_grade"] in {"A", "B", "C", "D"}
assert run.loc[0, "global_status"] == "WARNING"
assert len(metrics) == len(scored)

dashboard = pd.ExcelFile(DATA_CONTROL_CENTER_FILE)
assert dashboard.sheet_names == [
    "Quality Dashboard",
    "Quality Controls",
    "Quality Issues",
    "Quality History",
]
summary = pd.read_excel(
    DATA_CONTROL_CENTER_FILE,
    sheet_name="Quality Dashboard",
    header=1,
    nrows=1,
)
history = pd.read_excel(
    DATA_CONTROL_CENTER_FILE,
    sheet_name="Quality History",
)
assert summary.loc[0, "run_id"] == run_id
assert float(summary.loc[0, "global_quality_score"]) == float(run.loc[0, "global_quality_score"])
assert run_id in set(history["run_id"])

print(run[["run_id", "datasets_checked", "records_checked", "global_quality_score", "global_quality_grade", "global_status"]].to_string(index=False))
print("Résultat : PASS")


                   run_id  datasets_checked  records_checked  global_quality_score global_quality_grade global_status
DQ-20261023-115124-370379                 4           643506                  99.4                    A       WARNING
Résultat : PASS


In [3]:
controls.pivot(index="dataset_id", columns="control_name", values="status")[
    REQUIRED_CONTROLS
]


control_name,Valeurs manquantes,Doublons,Valeurs aberrantes,Formats incorrects,Cohérence des devises,Chronologie
dataset_id,,,,,,
DS-002,PASS,PASS,NOT_APPLICABLE,PASS,WARNING,NOT_APPLICABLE
DS-003,WARNING,PASS,WARNING,NOT_APPLICABLE,NOT_APPLICABLE,PASS
DS-004,PASS,PASS,PASS,NOT_APPLICABLE,PASS,PASS
DS-005,PASS,PASS,PASS,NOT_APPLICABLE,PASS,PASS


In [4]:
from openpyxl import load_workbook

from src.config import REPORTING_CURRENCY
from src.preparation.quality_treatment import (
    CHF_UNIVERSE,
    MARKET_PRICE_EXCLUSIONS,
    MARKET_PRICES_CHF,
    QUOTE_CURRENCY_FILE,
    SARON_TREATED,
    assign_quote_scale,
)
from src.quality.quality_config import (
    OHLC_ABSOLUTE_TOLERANCE,
    OHLC_RELATIVE_TOLERANCE,
)

connection = sqlite3.connect(DATABASE_FILE)
source_prices = pd.read_sql("SELECT COUNT(*) AS n FROM market_prices", connection).iloc[0, 0]
risk_free_table = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type = 'table' AND name = 'risk_free_rate'",
    connection,
)
raw_symbols = set(pd.read_sql("SELECT DISTINCT symbol FROM market_prices", connection)["symbol"])
master_symbols = pd.read_sql(
    "SELECT yahoo_symbol, listing_currency FROM securities_master",
    connection,
)
fx_rates = pd.read_sql("SELECT series_key, date, value FROM fx_rates", connection)
connection.close()

from src.quality.quality_checks import snb_quote_table

master_symbols["listing_currency"] = (
    master_symbols["listing_currency"].astype(str).str.strip().str.upper()
)
published_currencies = set(snb_quote_table(fx_rates)["listing_currency"])
currencies_without_series = (
    set(master_symbols["listing_currency"]) - {REPORTING_CURRENCY} - published_currencies
)

prices = pd.read_csv(MARKET_PRICES_CHF)
exclusions = pd.read_csv(MARKET_PRICE_EXCLUSIONS, dtype=str, keep_default_na=False)
universe = pd.read_csv(CHF_UNIVERSE, dtype=str, keep_default_na=False)

assert len(prices) + len(exclusions) == int(source_prices)
assert set(exclusions["reason"]) <= {
    "missing_close",
    "negative_price",
    "outside_high_low",
    "symbol_not_in_master",
    "no_snb_series",
    "missing_monthly_fx",
    "quote_currency_missing",
    "quote_currency_unmatched",
}

numeric = prices.copy()
for column in ("open", "high", "low", "close", "volume", "quote_scale", "chf_per_unit", "close_chf"):
    numeric[column] = pd.to_numeric(numeric[column], errors="coerce")
assert numeric["close"].notna().all()
assert numeric["close_chf"].notna().all()
assert not (numeric[["open", "high", "low", "close", "volume"]] < 0).any(axis=1).any()
span = (numeric["high"] - numeric["low"]).abs()
tolerance = (span * OHLC_RELATIVE_TOLERANCE).clip(lower=OHLC_ABSOLUTE_TOLERANCE)
comparable = numeric[["open", "high", "low", "close"]].notna().all(axis=1)
outside = comparable & (
    (numeric["open"] > numeric["high"] + tolerance)
    | (numeric["open"] < numeric["low"] - tolerance)
    | (numeric["close"] > numeric["high"] + tolerance)
    | (numeric["close"] < numeric["low"] - tolerance)
    | (numeric["high"] < numeric["low"])
)
assert int(outside.sum()) == 0
quote_currencies = pd.read_csv(QUOTE_CURRENCY_FILE, dtype=str, keep_default_na=False)
used = prices[["symbol", "quote_currency"]].drop_duplicates()
compared = used.merge(quote_currencies, on="symbol", suffixes=("_used", "_yahoo"))
assert len(compared) == len(used)
assert compared["quote_currency_used"].eq(compared["quote_currency_yahoo"]).all()
expected_scale = assign_quote_scale(prices)
assert numeric["quote_scale"].eq(expected_scale.astype(float)).all()
rebuilt = numeric["close"] / numeric["quote_scale"] * numeric["chf_per_unit"]
assert (numeric["close_chf"] - rebuilt).abs().max() < 1e-6
assert not set(numeric["listing_currency"].dropna()).intersection(currencies_without_series)

excluded = universe[universe["chf_status"].eq("excluded")]
included = universe[universe["chf_status"].eq("included")]
assert len(universe) == len(master_symbols)
assert set(excluded["listing_currency"]) == currencies_without_series
chf_names = set(
    master_symbols.loc[
        master_symbols["listing_currency"].eq(REPORTING_CURRENCY),
        "yahoo_symbol",
    ]
)
assert chf_names <= set(included["yahoo_symbol"])
assert int(included["clean_price_rows"].astype(int).sum()) == len(prices)
without_prices = set(included.loc[included["clean_price_rows"].eq("0"), "yahoo_symbol"])
expected_without_prices = set(
    master_symbols.loc[
        ~master_symbols["yahoo_symbol"].isin(raw_symbols)
        & master_symbols["listing_currency"].isin(set(included["listing_currency"])),
        "yahoo_symbol",
    ]
)
assert without_prices == expected_without_prices
assert without_prices.isdisjoint(raw_symbols)

assert risk_free_table.empty
assert not (SARON_TREATED.parent / "risk_free_rate.csv").exists()

saron = pd.read_csv(SARON_TREATED, dtype=str, keep_default_na=False)
assert set(saron["series_id"]) == {"SARON"}
assert set(saron["rate_currency"]) == {REPORTING_CURRENCY}
assert saron["rate_note"].str.contains("CHF risk-free rate").all()
assert not saron["series_id"].eq("DGS3MO").any()
assert int(saron["value"].eq("").sum()) == 0
assert saron["date"].min() == "2009-01"

workbook = load_workbook(DATA_CONTROL_CENTER_FILE)
dashboard = workbook["Quality Dashboard"]
history = workbook["Quality History"]
assert len(dashboard._charts) == 1
assert len(history._charts) == 1
kpi = dashboard._charts[0]
trend = history._charts[0]
assert kpi.anchor._from.row >= 15
assert kpi.anchor.ext.cx / 360000 >= 22
assert kpi.anchor.ext.cy / 360000 >= 12
assert kpi.title.overlay is False
assert kpi.layout.manualLayout.y >= 0.15
assert trend.anchor._from.row >= history.max_row
assert trend.anchor.ext.cx / 360000 >= 22
assert trend.title.overlay is False

print("Fichiers traités et graphiques : PASS")


Fichiers traités et graphiques : PASS
